# Task 3, Step 2: Fixed Prefix Banks and the Pretrained-Draft Baseline

**Prefix banks.** The course script `setup_data.py --prefix-banks` passes `generator=` to `model.generate`, which the
pinned transformers 4.57.1 rejects ("model_kwargs are not used: ['generator']"). `src/prefix_banks.py` reproduces the
script's protocol -- the same 120 prompts and ids, per-prompt seeds 6304 + 10007 j (+123 for the draft), the same
position-sampling RNG strings and record schema, temperature 1 / top-p 1 / no top-k, 192-token responses -- with our
own seeded KV-cache sampling loop. Banks: `teacher` (target), `original_draft`, and `exact_spec_original_gamma4`
(our exact sampler, original draft, gamma = 4). Teacher and speculative banks sample the same target-prefix law and
serve as a distributional consistency control.

**Baseline benchmark.** Target-only vs exact speculative sampling with the unchanged 0.5B draft, gamma in {2, 4},
two paired repetitions, 120 held-out prompts, EOS-aware 64-new-token cap, adjacent target-only timing for every run
(order counterbalanced), plus a fixed-64-token ignore-EOS throughput diagnostic on 24 mathematics prompts and the
isolated draft / target decode-step costs at several context lengths.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task3_speculative")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2" or _ver("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5", "peft>=0.13"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task3", need_public=True, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from shared.src.io import read_jsonl, write_jsonl, save_json
from shared.src.plotting import save_figure
from task3_speculative.src import prefix_banks as PB
from task3_speculative.src.distill import load_models
from task3_speculative.src.spec_eval import run_benchmark, isolated_step_costs

M = load_models()
tok, target, draft = M["tok"], M["target"], M["draft"]
BANKS = ctx.checkpoints / "banks"
BANKS.mkdir(exist_ok=True)
OUT = ctx.results / "spec"
OUT.mkdir(exist_ok=True)

In [ ]:
for bank, kw in [("teacher", dict(model=target)), ("original_draft", dict(model=draft)),
                 ("exact_spec_original_gamma4", dict(model=target, draft=draft, gamma=4))]:
    f = BANKS / f"{bank}.jsonl.gz"
    if not f.exists():
        write_jsonl(f, PB.build_bank(bank, tok, producer=f"{bank} (Qwen2.5 instruct, local sampler)", **kw))
    print(bank, len(read_jsonl(f)), "records")

## Distributional consistency: teacher bank vs exact-speculative bank
Both should follow the target prefix law. Compared per prompt: response length, target NLL of the sampled tokens,
and the original draft's expected acceptance 1 - TV at the saved positions (permutation test over prompts).

In [ ]:
tb, sb = read_jsonl(BANKS / "teacher.jsonl.gz"), read_jsonl(BANKS / "exact_spec_original_gamma4.jsonl.gz")
nll = pd.DataFrame(PB.sampled_token_nll(target, tok, tb + sb))
lens = pd.DataFrame([dict(prompt_id=r["prompt_id"], bank=r["bank"], length=r["source_length"]) for r in tb + sb]).drop_duplicates()
cons_m = pd.DataFrame(PB.bank_metrics(target, {"original": draft}, tok, tb + sb))
per_prompt = pd.concat([lens.pivot(index="prompt_id", columns="bank", values="length").add_prefix("len_"),
                        nll.groupby(["prompt_id", "bank"]).nll.mean().unstack().add_prefix("nll_"),
                        cons_m.groupby(["prompt_id", "bank"]).exp_accept.mean().unstack().add_prefix("acc_")], axis=1)

def perm_test(a, b, n=10000, seed=6304):
    rng = np.random.default_rng(seed)
    d = np.asarray(a) - np.asarray(b)
    obs = d.mean()
    null = (d[None, :] * rng.choice([-1, 1], size=(n, len(d)))).mean(1)
    return float(obs), float((np.abs(null) >= abs(obs)).mean())

rows = []
for m in ("len", "nll", "acc"):
    x = per_prompt[[f"{m}_teacher", f"{m}_exact_spec_original_gamma4"]].dropna()
    d, p = perm_test(x.iloc[:, 0], x.iloc[:, 1])
    rows.append(dict(metric=m, teacher=x.iloc[:, 0].mean(), speculative=x.iloc[:, 1].mean(), diff=d, perm_p=p, n_prompts=len(x)))
consistency = pd.DataFrame(rows)
consistency.to_csv(OUT / "teacher_vs_speculative_consistency.csv", index=False)
consistency

## Isolated decode-step costs (c and v_gamma)

In [ ]:
iso = pd.DataFrame(isolated_step_costs(target, draft, tok))
iso.to_csv(OUT / "isolated_step_costs_original.csv", index=False)
iso

## Speculative benchmark with the original draft (EOS-aware, 64-token cap)

In [ ]:
bench_path = OUT / "benchmark_eos64.jsonl"
run_benchmark(target, {"original": draft}, tok, bench_path)
fix_path = OUT / "benchmark_fixed64_ignore_eos.jsonl"
from task3_speculative.src.data import test_prompts
math24 = set([p["pid"] for p in test_prompts() if p["domain"] == "math"][:24])   # fixed: first 24 GSM8K test ids
run_benchmark(target, {"original": draft}, tok, fix_path, ignore_eos=True, prompt_filter=lambda p: p["pid"] in math24)

In [ ]:
b = pd.DataFrame(read_jsonl(bench_path))
cols = ["acc_per_examined", "acc_per_proposed", "mean_accepted_per_verification", "target_passes_per_token", "draft_forward_passes",
        "n_out", "eos", "truncated", "tokens_per_s", "base_tokens_per_s", "speedup_tps", "peak_allocated_mb", "base_peak_allocated_mb"]
b[b.draft == "original"].groupby(["gamma", "workload"])[cols].mean().round(3)

In [ ]:
dec = b.groupby("gamma")[["prefill_ms", "draft_ms", "verify_ms", "accept_ms", "rollback_ms", "host_overhead_ms", "total_ms", "base_total_ms"]].mean()
dec.round(1)